This will fully run the simulations, analyses and plots for s1. this was run on a cluster and for the smiulations it would be recommended to run there. Or speed up the algorithm to be gpu accelerated or jax based. 

The longest condition performs 100,000/0.01×10=100 million BAOAB steps per trajectory, with 1000 trajectories and 10 replicates.

In [ ]:
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from numba import njit, prange


# =============================================================================
# Configuration
# =============================================================================

OUTPUT_DIR = Path(
    "./output/fig-s1"
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_REPLICATES = 10
N_TRAJECTORIES = 1000

DT = 0.01
GAMMA = 1.0
INITIAL_VELOCITY = 0.0

# Sequential cooling schedule used in the original simulation.
KBT_ARRAY = np.linspace(1.0, 0.1, 10)

# Duration spent at EACH temperature in the cooling schedule.
MAX_TIME_ARRAY = np.array(
    [
        0.02,
        0.05,
        0.1,
        0.2,
        0.5,
        1,
        2,
        5,
        10,
        20,
        50,
        100,
        200,
        500,
        1000,
        2000,
        5000,
        10000,
        20000,
        50000,
        100000,
    ],
    dtype=np.float64,
)

# Total number of integration steps over the complete 10-temperature schedule.
VALUES = (
    MAX_TIME_ARRAY
    * len(KBT_ARRAY)
    / DT
)

# Reproducible independent seeds.
INITIAL_POSITION_SEED_BASE = 10_000
SIMULATION_SEED_BASE = 1_000_000


# =============================================================================
# Analytical potential
# =============================================================================

def potential(x):
    """
    Return potential energy V(x) and force F(x).

    V(x) = 4 [
        x^8
        + 1.6 exp(-80 x^2)
        + 0.2 exp(-80 (x - 0.5)^2)
        + 0.5 exp(-40 (x + 0.5)^2)
    ]
    """

    x = np.asarray(x)

    V = 4.0 * (
        x**8
        + 1.6 * np.exp(-80.0 * x**2)
        + 0.2 * np.exp(-80.0 * (x - 0.5)**2)
        + 0.5 * np.exp(-40.0 * (x + 0.5)**2)
    )

    F = -4.0 * (
        8.0 * x**7
        - 256.0 * np.exp(-80.0 * x**2) * x
        - 32.0 * np.exp(-80.0 * (x - 0.5)**2) * (x - 0.5)
        - 40.0 * np.exp(-40.0 * (x + 0.5)**2) * (x + 0.5)
    )

    return V, F


def force(x):
    return potential(x)[1]


# =============================================================================
# Locate stationary points / cluster boundaries
# =============================================================================

def find_stationary_points(
    x_min=-2.0,
    x_max=2.0,
    n_grid=100_001,
):
    """
    Robustly locate all roots of F(x)=0 in the supplied interval.

    This replaces the old multi-start fsolve call, which can return duplicate
    roots depending on the initial guesses.
    """

    grid = np.linspace(
        x_min,
        x_max,
        n_grid,
    )

    force_values = force(grid)

    roots = []

    sign_change_indices = np.where(
        np.signbit(force_values[:-1])
        != np.signbit(force_values[1:])
    )[0]

    for index in sign_change_indices:
        root = brentq(
            lambda z: float(force(z)),
            grid[index],
            grid[index + 1],
        )

        roots.append(root)

    roots = np.asarray(
        roots,
        dtype=np.float64,
    )

    # Remove any numerical duplicates.
    roots = np.unique(
        np.round(roots, decimals=12)
    )

    if len(roots) != 7:
        raise RuntimeError(
            "Expected 7 stationary points for the quad-well potential, "
            f"but found {len(roots)}:\n{roots}"
        )

    return roots


ROOTS = find_stationary_points()

# The original cluster definition uses roots 1, 3 and 5 as barriers.
CLUSTER_BOUNDARIES = np.array(
    [
        ROOTS[1],
        ROOTS[3],
        ROOTS[5],
    ],
    dtype=np.float64,
)

print("Stationary points:")
for index, root in enumerate(ROOTS):
    print(f"  root {index}: {root:.6f}")

print("\nCluster boundaries:")
for boundary in CLUSTER_BOUNDARIES:
    print(f"  {boundary:.6f}")


# =============================================================================
# Equilibrium initial-position sampling
# =============================================================================

def build_equilibrium_cdf(
    kBT=1.0,
    x_min=-2.0,
    x_max=2.0,
    n_grid=200_001,
):
    """
    Numerically construct the equilibrium positional distribution

        p(x) ∝ exp[-V(x) / kBT]

    for inverse-CDF sampling.
    """

    x_grid = np.linspace(
        x_min,
        x_max,
        n_grid,
        dtype=np.float64,
    )

    V, _ = potential(x_grid)

    log_probability = -V / kBT

    # Numerical stabilization.
    log_probability -= np.max(log_probability)

    probability = np.exp(
        log_probability
    )

    dx = x_grid[1] - x_grid[0]

    # Trapezoidal cumulative integral.
    cdf = np.zeros_like(
        probability
    )

    cdf[1:] = np.cumsum(
        0.5
        * (
            probability[:-1]
            + probability[1:]
        )
        * dx
    )

    cdf /= cdf[-1]

    return x_grid, cdf


EQUILIBRIUM_X, EQUILIBRIUM_CDF = build_equilibrium_cdf(
    kBT=KBT_ARRAY[0]
)


def sample_equilibrium_positions(
    n_samples,
    seed,
):
    """
    Reproducibly sample positions from equilibrium at kBT=1.
    """

    rng = np.random.default_rng(
        seed
    )

    uniform_samples = rng.random(
        n_samples
    )

    positions = np.interp(
        uniform_samples,
        EQUILIBRIUM_CDF,
        EQUILIBRIUM_X,
    )

    return positions.astype(
        np.float64
    )


# =============================================================================
# Cluster assignment
# =============================================================================

def assign_clusters(values):
    """
    Assign coordinates to the four wells using the three barrier positions.

    Cluster 0: x < boundary 0
    Cluster 1: boundary 0 <= x < boundary 1
    Cluster 2: boundary 1 <= x < boundary 2
    Cluster 3: x >= boundary 2
    """

    return np.digitize(
        np.asarray(values),
        CLUSTER_BOUNDARIES,
        right=False,
    )


# =============================================================================
# Numba-compiled BAOAB dynamics
# =============================================================================

@njit(inline="always")
def force_scalar(x):
    """
    Scalar force implementation for the Numba simulation kernel.
    """

    return -4.0 * (
        8.0 * x**7
        - 256.0 * math.exp(-80.0 * x**2) * x
        - 32.0
        * math.exp(-80.0 * (x - 0.5)**2)
        * (x - 0.5)
        - 40.0
        * math.exp(-40.0 * (x + 0.5)**2)
        * (x + 0.5)
    )


@njit
def simulate_single_trajectory(
    initial_position,
    steps_per_temperature,
    dt,
    gamma,
    kBT_array,
    random_seed,
):
    """
    Run one complete cooling trajectory.

    The same duration is spent at each successive temperature.

    Only the true final position is retained because the final figure does
    not use the intermediate trajectory.
    """

    np.random.seed(
        random_seed
    )

    x = initial_position
    v = INITIAL_VELOCITY

    c1 = math.exp(
        -gamma * dt
    )

    noise_prefactor = math.sqrt(
        1.0 - c1 * c1
    )

    for temperature_index in range(
        len(kBT_array)
    ):
        kBT = kBT_array[
            temperature_index
        ]

        c2 = (
            noise_prefactor
            * math.sqrt(kBT)
        )

        for _ in range(
            steps_per_temperature
        ):
            # B
            F = force_scalar(x)
            v += 0.5 * F * dt

            # A
            x += 0.5 * v * dt

            # O
            v = (
                c1 * v
                + np.random.normal() * c2
            )

            # A
            x += 0.5 * v * dt

            # B
            F = force_scalar(x)
            v += 0.5 * F * dt

    return x


@njit(parallel=True)
def simulate_ensemble(
    initial_positions,
    steps_per_temperature,
    dt,
    gamma,
    kBT_array,
    seed_base,
):
    """
    Run the complete cooling protocol for an ensemble of starting positions.

    Each trajectory receives its own deterministic random seed.
    """

    n_trajectories = len(
        initial_positions
    )

    final_positions = np.empty(
        n_trajectories,
        dtype=np.float64,
    )

    for trajectory_index in prange(
        n_trajectories
    ):
        final_positions[
            trajectory_index
        ] = simulate_single_trajectory(
            initial_positions[
                trajectory_index
            ],
            steps_per_temperature,
            dt,
            gamma,
            kBT_array,
            seed_base
            + trajectory_index,
        )

    return final_positions


# =============================================================================
# Compile simulation kernel before the main calculation
# =============================================================================

print("\nCompiling BAOAB kernel...")

_ = simulate_ensemble(
    np.array(
        [0.0],
        dtype=np.float64,
    ),
    1,
    DT,
    GAMMA,
    np.array(
        [1.0],
        dtype=np.float64,
    ),
    12345,
)

print("Compilation complete.")


# =============================================================================
# Run replicate cooling calculations and calculate populations
# =============================================================================

n_times = len(
    MAX_TIME_ARRAY
)

clusters_data = np.zeros(
    (
        N_REPLICATES,
        4,
        n_times,
    ),
    dtype=np.float64,
)

macrostate_A_data = np.zeros(
    (
        N_REPLICATES,
        n_times,
    ),
    dtype=np.float64,
)

macrostate_B_data = np.zeros(
    (
        N_REPLICATES,
        n_times,
    ),
    dtype=np.float64,
)


for replicate in range(
    N_REPLICATES
):
    print(
        f"\nReplicate "
        f"{replicate + 1}/{N_REPLICATES}"
    )

    # -------------------------------------------------------------------------
    # Generate one reproducible equilibrium starting ensemble.
    #
    # This SAME starting ensemble is used for every cooling duration within
    # this replicate.
    # -------------------------------------------------------------------------

    initial_positions = (
        sample_equilibrium_positions(
            n_samples=N_TRAJECTORIES,
            seed=(
                INITIAL_POSITION_SEED_BASE
                + replicate
            ),
        )
    )

    initial_assignments = (
        assign_clusters(
            initial_positions
        )
    )

    initial_populations = np.bincount(
        initial_assignments,
        minlength=4,
    ).astype(
        np.float64
    )

    if np.any(
        initial_populations == 0
    ):
        raise RuntimeError(
            f"Replicate {replicate:02d} has an empty initial cluster:\n"
            f"{initial_populations}"
        )

    initial_macrostate_A = (
        initial_populations[0]
        + initial_populations[1]
    )

    initial_macrostate_B = (
        initial_populations[2]
        + initial_populations[3]
    )

    print(
        "  initial populations:",
        initial_populations.astype(int),
    )

    # -------------------------------------------------------------------------
    # Independently run each cooling-duration experiment from the SAME
    # starting ensemble.
    # -------------------------------------------------------------------------

    for time_index, max_time in enumerate(
        MAX_TIME_ARRAY
    ):
        steps_per_temperature = int(
            round(
                max_time / DT
            )
        )

        total_steps = (
            steps_per_temperature
            * len(KBT_ARRAY)
        )

        print(
            f"  {time_index + 1:02d}/{n_times}  "
            f"max_time={max_time:g}  "
            f"total_steps={total_steps:,}"
        )

        # Give every (replicate, cooling duration, trajectory) an independent
        # reproducible random stream.
        seed_base = (
            SIMULATION_SEED_BASE
            + replicate * 100_000
            + time_index * 2_000
        )

        final_positions = (
            simulate_ensemble(
                initial_positions,
                steps_per_temperature,
                DT,
                GAMMA,
                KBT_ARRAY,
                seed_base,
            )
        )

        if not np.all(
            np.isfinite(
                final_positions
            )
        ):
            raise RuntimeError(
                "Non-finite final positions encountered for "
                f"replicate {replicate:02d}, "
                f"max_time={max_time:g}."
            )

        final_assignments = (
            assign_clusters(
                final_positions
            )
        )

        final_populations = np.bincount(
            final_assignments,
            minlength=4,
        ).astype(
            np.float64
        )

        # ---------------------------------------------------------------------
        # Cluster populations normalized by their respective starting
        # populations.
        # ---------------------------------------------------------------------

        clusters_data[
            replicate,
            :,
            time_index,
        ] = (
            final_populations
            / initial_populations
        )

        # ---------------------------------------------------------------------
        # Macrostate A = clusters 0 + 1
        # Macrostate B = clusters 2 + 3
        # ---------------------------------------------------------------------

        final_macrostate_A = (
            final_populations[0]
            + final_populations[1]
        )

        final_macrostate_B = (
            final_populations[2]
            + final_populations[3]
        )

        macrostate_A_data[
            replicate,
            time_index,
        ] = (
            final_macrostate_A
            / initial_macrostate_A
        )

        macrostate_B_data[
            replicate,
            time_index,
        ] = (
            final_macrostate_B
            / initial_macrostate_B
        )


# =============================================================================
# Calculate replicate statistics
# =============================================================================

cluster_means = np.mean(
    clusters_data,
    axis=0,
)

cluster_stds = np.std(
    clusters_data,
    axis=0,
)

mean_A = np.mean(
    macrostate_A_data,
    axis=0,
)

std_A = np.std(
    macrostate_A_data,
    axis=0,
)

mean_B = np.mean(
    macrostate_B_data,
    axis=0,
)

std_B = np.std(
    macrostate_B_data,
    axis=0,
)


# =============================================================================
# Potential data for panel A
# =============================================================================

x = np.linspace(
    -2.0,
    2.0,
    1000,
)

V, F = potential(x)


# =============================================================================
# Build combined figure
# =============================================================================

fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 14),
)


# =============================================================================
# Panel A: potential
# =============================================================================

ax = axes[0]

ax.plot(
    x[x < ROOTS[3]],
    V[x < ROOTS[3]],
    color="orange",
    lw=5,
)

ax.plot(
    x[
        (x >= ROOTS[3])
        & (x < ROOTS[5])
    ],
    V[
        (x >= ROOTS[3])
        & (x < ROOTS[5])
    ],
    color="blue",
    lw=5,
)

ax.plot(
    x[x >= ROOTS[5]],
    V[x >= ROOTS[5]],
    color="blue",
    lw=5,
)

ax.fill_between(
    x,
    V,
    7,
    where=(
        x < ROOTS[1]
    ),
    color="lightsalmon",
    alpha=0.5,
    label="Cluster 0",
)

ax.fill_between(
    x,
    V,
    7,
    where=(
        (x >= ROOTS[1])
        & (x < ROOTS[3])
    ),
    color="coral",
    alpha=0.5,
    label="Cluster 1",
)

ax.fill_between(
    x,
    V,
    7,
    where=(
        (x >= ROOTS[3])
        & (x < ROOTS[5])
    ),
    color="steelblue",
    alpha=0.5,
    label="Cluster 2",
)

ax.fill_between(
    x,
    V,
    7,
    where=(
        x >= ROOTS[5]
    ),
    color="lightblue",
    alpha=0.5,
    label="Cluster 3",
)

ax.plot(
    [],
    [],
    color="orange",
    label="Macrostate A",
)

ax.plot(
    [],
    [],
    color="blue",
    label="Macrostate B",
)

ax.set_xlabel(
    "x",
    fontsize=20,
)

ax.set_ylabel(
    "V(x)",
    fontsize=20,
)

ax.set_xlim(
    -1.3,
    1.65,
)

ax.set_ylim(
    0,
    7,
)

ax.legend(
    fontsize=16,
    loc="lower right",
)

ax.tick_params(
    labelsize=18
)

ax.set_title(
    "1D potential with four wells",
    fontsize=24,
    pad=15,
)

ax.text(
    -0.1,
    1.02,
    "A",
    transform=ax.transAxes,
    fontsize=36,
    va="bottom",
    ha="right",
)


# =============================================================================
# Panel B: normalized populations
# =============================================================================

ax = axes[1]

colors = [
    "lightsalmon",
    "coral",
    "steelblue",
    "lightblue",
]

for cluster_index in range(4):
    mean = cluster_means[
        cluster_index
    ]

    std = cluster_stds[
        cluster_index
    ]

    ax.plot(
        VALUES,
        mean,
        "--",
        color=colors[
            cluster_index
        ],
        label=f"Cluster {cluster_index}",
    )

    ax.fill_between(
        VALUES,
        mean - std,
        mean + std,
        color=colors[
            cluster_index
        ],
        alpha=0.3,
    )


ax.plot(
    VALUES,
    mean_A,
    "-",
    color="orange",
    label="Macrostate A",
)

ax.fill_between(
    VALUES,
    mean_A - std_A,
    mean_A + std_A,
    color="orange",
    alpha=0.3,
)

ax.plot(
    VALUES,
    mean_B,
    "-",
    color="blue",
    label="Macrostate B",
)

ax.fill_between(
    VALUES,
    mean_B - std_B,
    mean_B + std_B,
    color="blue",
    alpha=0.3,
)

ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Cooling Time (Steps)",
    fontsize=20,
)

ax.set_ylabel(
    "Fraction of Starting Population",
    fontsize=20,
)

ax.legend(
    fontsize=16
)

ax.tick_params(
    labelsize=18
)

ax.set_title(
    "Normalized population changes across cooling rates",
    fontsize=24,
    pad=15,
)

ax.text(
    -0.1,
    1.02,
    "B",
    transform=ax.transAxes,
    fontsize=36,
    va="bottom",
    ha="right",
)


# =============================================================================
# Save outputs
# =============================================================================

plt.tight_layout()

svg_path = (
    OUTPUT_DIR
    / "quad_well_combined.svg"
)

pdf_path = (
    OUTPUT_DIR
    / "quad_well_combined.pdf"
)

fig.savefig(
    svg_path,
    dpi=600,
)

fig.savefig(
    pdf_path,
    dpi=600,
)

print(
    f"\nSaved:\n"
    f"  {svg_path}\n"
    f"  {pdf_path}"
)

plt.show()